## This project will primarily use Python and SQL, with Pandas and DuckDB as the main libraries for data manipulation, transformation, and analysis.

Pandas provides flexible data handling for exploration and cleaning, while DuckDB allows efficient SQL queries directly on Parquet files.
This lightweight stack is sufficient for the dataset size.

## This notebook will be used for the exploratory data analysis, this notebook leads to no technical results.

In [ ]:
#Run this cell to import necessary libraries and set up data paths for the analysis. Do not edit this cell.
import pandas as pd
from pathlib import Path
import duckdb
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

data_path = Path("../data/raw")
visitors_path = (data_path / "visitors.parquet").as_posix()
sessions_path = (data_path / "sessions.parquet").as_posix()
events_path = (data_path / "questionnaire_events.parquet").as_posix()
answers_path = (data_path / "questionnaire_answers.parquet").as_posix()
questions_path = (data_path / "questionnaire_questions.parquet").as_posix()
outcomes_path = (data_path / "questionnaire_outcomes.parquet").as_posix()

tables = {
    "visitors": pd.read_parquet(visitors_path),
    "sessions": pd.read_parquet(sessions_path),
    "questionnaire_questions": pd.read_parquet(questions_path),
    "questionnaire_events": pd.read_parquet(events_path),
    "questionnaire_answers": pd.read_parquet(answers_path),
    "questionnaire_outcomes": pd.read_parquet(outcomes_path)
}


↓ We started by verifying that the data could be read correctly and checking for orphaned records across the related tables.

In [ ]:
##Test code to read all parquet files in the data directory

for file in data_path.glob("*.parquet"):
    df = pd.read_parquet(file)
    print(file.name, df.shape)
    display(df.head())

In [ ]:
## Test code to read all parquet files in the data directory

duckdb.sql(f"""
SELECT *
FROM read_parquet('{data_path.as_posix()}/*.parquet')
LIMIT 5
""").show()

↓ No orphaned records were found, and all datasets were successfully read. We can now proceed with the exploratory data analysis.

The results soon provided will not only be used for a future data clean but also to have a clear perspective over future analytical paths.

In [ ]:
## Check for orphaned records in child tables that do not have a corresponding parent record.

checks = [
    ("sessions", sessions_path, "visitors", visitors_path, "visitor_id"),
    ("questionnaire_events", events_path, "sessions", sessions_path, "session_id"),
    ("questionnaire_answers", answers_path, "sessions", sessions_path, "session_id"),
    ("questionnaire_answers", answers_path, "questionnaire_questions", questions_path, "question_id"),
    ("questionnaire_outcomes", outcomes_path, "sessions", sessions_path, "session_id")
]

for child, child_path, parent, parent_path, key in checks:
    n = duckdb.sql(f"""
        SELECT COUNT(*)
        FROM read_parquet('{child_path}') c
        LEFT JOIN read_parquet('{parent_path}') p USING ({key})
        WHERE p.{key} IS NULL
    """).fetchone()[0]

    print(f"{child} → {parent}: {n} orphelins")

↓ No duplicated rows in the provided datasets, and no missing relevant IDs.

In [ ]:
# Check for duplicates and missing relevant IDs in each table.

key_columns = {
    "visitors": ["visitor_id"],
    "sessions": ["session_id", "visitor_id"],
    "questionnaire_questions": ["question_id"],
    "questionnaire_events": ["event_id", "session_id", "visitor_id"],
    "questionnaire_answers": ["answer_id", "session_id", "visitor_id", "question_id"],
    "questionnaire_outcomes": ["session_id", "visitor_id"]
}

for name, df in tables.items():
    duplicates = df.duplicated().sum()
    missing_ids = df[key_columns[name]].isna().any(axis=1).sum()

    print(f"{name}:")
    print(f"  Exact duplicates: {duplicates}")
    print(f"  Rows with missing relevant IDs: {missing_ids}\n")

↓ No major inconsistencies were identified in the sessions related data.


In [ ]:
#Check for the number of completed questionnaires and the number of outcomes to see if they match.
duckdb.sql(f"""
SELECT
    (SELECT COUNT(*)
     FROM read_parquet('{events_path}')
     WHERE event_type = 'questionnaire_complete') AS completed_questionnaires,

    (SELECT COUNT(*)
     FROM read_parquet('{outcomes_path}')) AS outcomes
""").df()

In [ ]:
#Check if a session has multiple outcomes, which should not happen.
duckdb.sql(f"""
SELECT COUNT(*) AS sessions_with_multiple_outcomes
FROM (
    SELECT session_id
    FROM read_parquet('{outcomes_path}')
    GROUP BY session_id
    HAVING COUNT(*) > 1
)
""").df()

↓ After several checks on the sessions and outcomes tables, we identified visitors who completed multiple questionnaires and received different outcomes across sessions. This may reflect inconsistent self-reported data, repeated testing behavior, or genuine changes over time. We also identified visitors whose outcomes followed a coherent progression, such as no_current_indication → possible_risk → declared_diagnosed, suggesting that not all outcome changes should be treated as data-quality issues.

In [ ]:
#check for visitors with multiple outcomes which is not expected in the data. This is a data quality issue that needs to be addressed
duckdb.sql(f"""
SELECT
    visitor_id,
    COUNT(*) AS outcome_count
FROM read_parquet('{outcomes_path}')
GROUP BY visitor_id
HAVING COUNT(*) > 1
ORDER BY outcome_count DESC
""").df()

In [ ]:
#check for visitors with multiple completed sessions.
#They may have participated in multiple campaigns but they also may have provided different answers to the same questrinnaires.
#We must check if the outcomes check and if not it must be adressed as a data quality issue.
duckdb.sql(f"""
SELECT
    o.visitor_id,
    COUNT(DISTINCT o.session_id) AS completed_sessions,
    LIST(DISTINCT s.acquisition_source) AS sources,
    LIST(DISTINCT s.campaign_name) AS campaigns
FROM read_parquet('{outcomes_path}') o
JOIN read_parquet('{sessions_path}') s
    ON o.session_id = s.session_id
GROUP BY o.visitor_id
HAVING COUNT(DISTINCT o.session_id) > 1
ORDER BY completed_sessions DESC
""").df()

In [ ]:
#check for the total number of completed sessions across all visitors with multiple completed sessions.
duckdb.sql(f"""
SELECT SUM(completed_sessions) AS total_completed_sessions
FROM (
    SELECT visitor_id, COUNT(*) AS completed_sessions
    FROM read_parquet('{outcomes_path}')
    GROUP BY visitor_id
    HAVING COUNT(*) > 1
)
""").df()

In [ ]:
#Check for visitors with multiple outcomes but reccurrent outcomes, which may be expected in the data.
#Visitors may also have participated in multiple campaigns.
duckdb.sql(f"""
SELECT
    o.visitor_id,
    COUNT(*) AS outcome_count,
    LIST(DISTINCT o.outcome_category) AS outcomes,
    LIST(DISTINCT s.campaign_name) AS campaigns
FROM read_parquet('{outcomes_path}') o
JOIN read_parquet('{sessions_path}') s
    ON o.session_id = s.session_id
GROUP BY o.visitor_id
HAVING COUNT(*) > 1
   AND COUNT(DISTINCT o.outcome_category) = 1
ORDER BY outcome_count DESC
""").df()

In [ ]:
#Check for visitors with multiple distinct outcomes, which is not expected in the data. This is a data quality issue that needs to be addressed.
#Data may not be coherent in the case where a visitor may have been diagnosed after a first session and then after a second session they may have provided a different answer. Must be adressed.
duckdb.sql(f"""
SELECT
    visitor_id,
    COUNT(DISTINCT outcome_category) AS distinct_outcomes,
    LIST(DISTINCT outcome_category) AS outcomes
FROM read_parquet('{outcomes_path}')
GROUP BY visitor_id
HAVING COUNT(DISTINCT outcome_category) > 1
ORDER BY distinct_outcomes DESC
""").df()

In [ ]:
#Check for visitors with multiple outcomes but coherent evolution, which may be expected in the data.
duckdb.sql(f"""
WITH ordered AS (
    SELECT *,
        CASE outcome_category
            WHEN 'no_current_indication' THEN 1
            WHEN 'possible_risk' THEN 2
            WHEN 'declared_diagnosed' THEN 3
        END AS level,
        LAG(level) OVER (
            PARTITION BY visitor_id
            ORDER BY completed_at
        ) AS previous_level
    FROM read_parquet('{outcomes_path}')
)

SELECT
    visitor_id,
    LIST(outcome_category ORDER BY completed_at) AS evolution,
    LIST(STRFTIME(completed_at, '%d/%m/%Y') ORDER BY completed_at) AS dates
FROM ordered
GROUP BY visitor_id
HAVING COUNT(DISTINCT outcome_category) > 1
   AND SUM(CASE WHEN level < previous_level THEN 1 ELSE 0 END) = 0
ORDER BY visitor_id
""").df()

In [ ]:
#check for visitors with multiple outcomes but incoherent evolution, which is not expected in the data. This is a data quality issue that needs to be addressed.
duckdb.sql(f"""
WITH x AS (
    SELECT *,
        CASE outcome_category
            WHEN 'no_current_indication' THEN 1
            WHEN 'possible_risk' THEN 2
            WHEN 'declared_diagnosed' THEN 3
        END AS level,
        LAG(level) OVER (PARTITION BY visitor_id ORDER BY completed_at) AS prev
    FROM read_parquet('{outcomes_path}')
)

SELECT visitor_id,
       LIST(outcome_category ORDER BY completed_at) AS evolution,
       LIST(STRFTIME(completed_at, '%d/%m/%Y') ORDER BY completed_at) AS dates
FROM x
GROUP BY visitor_id
HAVING SUM(CASE WHEN prev = 3 AND level < 3 THEN 1 ELSE 0 END) > 0
ORDER BY visitor_id
""").df()

↓ The questionnaire and outcome data do not appear to contain any major internal inconsistencies.

In [ ]:
#checking for outcomes provided that do not match any completed questionnaire.
duckdb.sql(f"""
SELECT count(*) AS outcomes_without_complete_event
FROM read_parquet('{outcomes_path}') o
LEFT JOIN read_parquet('{events_path}') e
ON o.session_id = e.session_id
AND e.event_type = 'questionnaire_complete'
WHERE e.session_id IS NULL
""").df()

In [ ]:
#check for questionnaires completed that do not have a corresponding outcome.
duckdb.sql(f"""
SELECT count(*) AS events_complete_without_outcome
FROM read_parquet('{events_path}') e
LEFT JOIN read_parquet('{outcomes_path}') o
ON e.session_id = o.session_id
WHERE e.event_type = 'questionnaire_complete'
AND o.session_id IS NULL
""").df()

In [ ]:
#Check for questionnaires that were completed but did not have all 5 questions answered.
duckdb.sql(f"""
SELECT COUNT(*) AS falsely_completed
FROM (
    SELECT e.session_id
    FROM read_parquet('{events_path}') e
    LEFT JOIN read_parquet('{answers_path}') a
        ON e.session_id = a.session_id
    WHERE e.event_type = 'questionnaire_complete'
    GROUP BY e.session_id
    HAVING COUNT(DISTINCT a.question_number) < 5
)
""").df()

In [ ]:
#check for questionnaires that were completed but did not have all 5 questions viewed.
duckdb.sql(f"""
SELECT COUNT(*) AS completed_without_5_views
FROM (
    SELECT session_id
    FROM read_parquet('{events_path}')
    WHERE session_id IN (
        SELECT session_id
        FROM read_parquet('{events_path}')
        WHERE event_type = 'questionnaire_complete'
    )
    GROUP BY session_id
    HAVING COUNT(DISTINCT CASE
        WHEN event_type = 'question_view'
        THEN question_number
    END) < 5
)
""").df()

↓ The timestamps appear to be internally consistent, with no major chronological anomalies identified.

In [ ]:
#check for events that have started before the session start time, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS events_before_session
FROM read_parquet('{events_path}') e
JOIN read_parquet('{sessions_path}') s USING(session_id)
WHERE e.event_timestamp < s.session_started_at
""").df()

In [ ]:
#check for questionnaires that got an outcome before their supposed completion, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS answers_after_complete
FROM read_parquet('{answers_path}') a
JOIN read_parquet('{outcomes_path}') o USING(session_id)
WHERE a.answered_at > o.completed_at
""").df()

In [ ]:
#check for events that have a start time after their end time, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS invalid_order
FROM (
    SELECT session_id,
           MIN(CASE WHEN event_type='questionnaire_start' THEN event_timestamp END) AS start_time,
           MIN(CASE WHEN event_type='questionnaire_complete' THEN event_timestamp END) AS end_time
    FROM read_parquet('{events_path}')
    GROUP BY session_id
)
WHERE start_time > end_time
""").df()

↓ Gender and region contain non-standardized categorical values, including inconsistent casing, abbreviations, and accent variations, which create duplicate representations of the same category.

In [ ]:
#checking for unusual device types in the sessions table, which may indicate data quality issues.
duckdb.sql(f"""
SELECT DISTINCT device_type FROM read_parquet('{sessions_path}');
""").show()

In [ ]:
#checking for false gender values in the sessions table, which may indicate data quality issues.
display(duckdb.sql(f"""
SELECT DISTINCT gender
FROM read_parquet('{visitors_path}')
ORDER BY gender
""").df())

In [ ]:
#checking for incorrect regions in the sessions table, which may indicate data quality issues.
display(duckdb.sql(f"""
SELECT DISTINCT region
FROM read_parquet('{visitors_path}')
ORDER BY region
""").df())

↓ All the answers provided are allowed.

In [ ]:
#Checking if all the questions have the same allowed answers, which is expected in the data.
display(duckdb.sql(f"""
SELECT DISTINCT question_number, answer_value
FROM read_parquet('{answers_path}')
ORDER BY question_number, answer_value
""").df())

↓ All the provided data respects the time range expected.

In [ ]:
#cehcking if all the provided data respects the expected time range of the study, which is from June 1st, 2026 to September 1st, 2026.
duckdb.sql(f"""
SELECT
    (SELECT COUNT(*) FROM read_parquet('{sessions_path}')
     WHERE session_started_at < '2026-06-01' OR session_started_at >= '2026-09-01') AS invalid_sessions,

    (SELECT COUNT(*) FROM read_parquet('{events_path}')
     WHERE event_timestamp < '2026-06-01' OR event_timestamp >= '2026-09-01') AS invalid_events,

    (SELECT COUNT(*) FROM read_parquet('{answers_path}')
     WHERE answered_at < '2026-06-01' OR answered_at >= '2026-09-01') AS invalid_answers,

    (SELECT COUNT(*) FROM read_parquet('{outcomes_path}')
     WHERE completed_at < '2026-06-01' OR completed_at >= '2026-09-01') AS invalid_outcomes
""").df()

↓ Birth years appear consistent, ranging from approximately 16 to 96 years old at the time of questionnaire completion.

In [ ]:
#checking for extreme birth years in the visitors table, which may indicate data quality issues.
duckdb.sql(f"""
SELECT
    MIN(birth_year) AS min_birth_year,
    MAX(birth_year) AS max_birth_year
FROM read_parquet('{visitors_path}')
""").df()

↓ The last checkings on visitor data revealed that many visitors did not provide enough information about themselves which could influence future analysis, i came up with two solutions, we either delete them in the data processing phase or flag them as if they prefered not to provide their personnal data and include them differently on the analysis which is my prefered approach.

In [ ]:
#checking for visitors with incomplete data, which is not expected in the data.
duckdb.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE visitor_id IS NULL) AS visitor_id_nulls,
    COUNT(*) FILTER (WHERE birth_year IS NULL) AS birth_year_nulls,
    COUNT(*) FILTER (WHERE gender IS NULL) AS gender_nulls,
    COUNT(*) FILTER (WHERE region IS NULL) AS region_nulls,
    COUNT(*) FILTER (WHERE first_seen_at IS NULL) AS first_seen_at_nulls
FROM read_parquet('{visitors_path}')
""").df()

In [ ]:
#checking for visitors with all personal data fields incomplete, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS visitors_with_incomplete_data
FROM (
    SELECT
        (birth_year IS NULL)::INT +
        (gender IS NULL)::INT +
        (region IS NULL)::INT AS null_count
    FROM read_parquet('{visitors_path}')
)
WHERE null_count > 2
""").df()

## EDA Conclusion

The exploratory analysis shows that the datasets are structurally consistent: no orphaned records, exact duplicates, missing critical identifiers, major timestamp anomalies, or inconsistencies between completed questionnaires, answers, views, and outcomes were identified.

The main data-quality issue concerns categorical standardization, particularly gender and region, where multiple representations of the same category exist, visitors with little to no personnal information provided should also be adressed seperately.

Visitors completing multiple questionnaires may also receive different outcomes across sessions. These cases should not automatically be treated as errors, as some show coherent evolution over time and may reflect repeated testing or changes in self-reported information.

Overall, the dataset requires limited cleaning, mainly categorical normalization, before proceeding to analytical transformations and deeper business analysis.